<a href="https://colab.research.google.com/github/K-Irungu/apache-spark-assignment/blob/main/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apache Spark Assignment: Chicago Taxi Trips Analysis
**Name:** Kevin Thuita Irungu  
**Student ID:** 668115  
**Date:** 24/09/2026

## Stage 1 - Initialize SparkSession and Load Data

### Install pyspark and requests

In [1]:
!pip install pyspark requests

### Download the dataset

In [2]:
import requests

url = "https://data.cityofchicago.org/resource/wrvz-psew.csv?$limit=200000&$order=trip_start_timestamp"
local_path = "chicago_taxi_trips.csv"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(local_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded file size: {len(response.content) / (1024*1024):.2f} MB")

Downloaded file size: 83.18 MB


### Create SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

# Stop any existing SparkSession
active_spark_session = SparkSession.getActiveSession()
if active_spark_session:
    print("Stopping existing SparkSession.")
    active_spark_session.stop()

# Ensure no SparkContext is running (more robust check)
if SparkContext._active_spark_context is not None:
    print("Stopping existing SparkContext directly.")
    SparkContext._active_spark_context.stop()

# Now create the new SparkSession
spark = SparkSession.builder \
    .appName("ChicagoTaxiAnalysis") \
    .getOrCreate()

spark

### Load csv into a Spark DataFrame

In [4]:
taxi_df = spark.read.csv(local_path, header=True, inferSchema=True)

### Print counts

In [5]:
row_count = taxi_df.count()
col_count = len(taxi_df.columns)

print(f"Total rows: {row_count}")
print(f"Total columns: {col_count}")

Total rows: 200000
Total columns: 23


## Stage 2 - Inspect the DataFrame

### Print the chema

In [6]:
taxi_df.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_cen

### List all column names

In [7]:
print(taxi_df.columns)

['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras', 'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude', 'pickup_centroid_longitude', 'pickup_centroid_location', 'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'dropoff_centroid_location']


### Display first 5 rows

In [8]:
taxi_df.show(5, truncate=False)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+--------------------+-------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+------+-----+-----+------+----------+------------+-------+------------------------+-------------------------+------------------------------------+-------------------------+--------------------------+-----------------------------------+
|trip_id                                 |taxi_id                                                                                                                         |trip_start_timestamp|trip_end_timestamp |trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|fare  |tips |tolls|extras|trip_total|payment_type|company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_cent

### Dataset Structure Reflection


The dataset contains 23 columns spanning several types: identifiers (`trip_id`, `taxi_id` as strings), timestamps (`trip_start_timestamp`, `trip_end_timestamp`), numeric trip metrics (`trip_seconds` as integer, `trip_miles`, `fare`, `tips`, `tolls`, `extras`, `trip_total` as doubles), geographic references (`pickup_community_area`, `dropoff_community_area` as integers, plus census tracts and centroid latitude/longitude/location fields), and categorical columns (`payment_type`, `company`).


For trip analysis, the most important columns are `trip_start_timestamp`, `trip_seconds`, `trip_miles`, `fare`/`trip_total`, and the pickup/dropoff community areas, since these drive time-based, distance-based, and location-based analysis. `inferSchema=True` correctly detected `trip_seconds` as integer and the timestamp columns as proper timestamp type without needing manual casting, and the sample rows show several trips have `NULL` values for census tract, community area, and company fields.

## Stage 3 - Identify and Handle Missing Data

### Count nulls per column

In [9]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = taxi_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in taxi_df.columns
])

null_counts.show(vertical=True, truncate=False)

-RECORD 0----------------------------
 trip_id                    | 0      
 taxi_id                    | 0      
 trip_start_timestamp       | 0      
 trip_end_timestamp         | 0      
 trip_seconds               | 3101   
 trip_miles                 | 5      
 pickup_census_tract        | 93490  
 dropoff_census_tract       | 100340 
 pickup_community_area      | 30354  
 dropoff_community_area     | 40599  
 fare                       | 0      
 tips                       | 0      
 tolls                      | 0      
 extras                     | 0      
 trip_total                 | 0      
 payment_type               | 0      
 company                    | 93843  
 pickup_centroid_latitude   | 30354  
 pickup_centroid_longitude  | 30354  
 pickup_centroid_location   | 30354  
 dropoff_centroid_latitude  | 40581  
 dropoff_centroid_longitude | 40581  
 dropoff_centroid_location  | 40581  



### Drop rows missing essential metrics

In [10]:
taxi_df = taxi_df.dropna(subset=["trip_total", "trip_miles"])

### Fill nulls in payment_type with UNKNOWN

In [11]:
taxi_df = taxi_df.fillna({"payment_type": "UNKNOWN"})

### Print new row count

In [12]:
new_row_count = taxi_df.count()
print(f"Row count after cleaning: {new_row_count}")

Row count after cleaning: 199995


### Missing Data Reflection

The columns with the highest null counts were `pickup_census_tract`, `dropoff_census_tract`, and `company`. This is because census tract data is only recorded when GPS coordinates fall within a mapped tract boundary, and many trips - especially those picked up or dropped off in areas without precise geolocation data, or trips logged by taxi companies that don't report a `company` name to the city — leave these fields blank. `pickup_community_area` and `dropoff_community_area` also had a notable number of nulls, likely for the same geolocation-reporting reasons.


In contrast, core trip metrics like `trip_total`, `trip_miles`, and `fare` had far fewer nulls, since these are essential, directly metered values that are almost always captured during the trip itself.

## Stage 4 - Filter and Transform





### Filter invalid trips

In [13]:
taxi_df = taxi_df.filter((col("trip_miles") > 0) & (col("trip_seconds") > 0))

filtered_row_count = taxi_df.count()
print(f"Row count after filtering invalid trips: {filtered_row_count}")

Row count after filtering invalid trips: 124797


### Select 15 columns

In [14]:
taxi_df = taxi_df.select(
    "trip_id", "taxi_id", "trip_start_timestamp", "trip_end_timestamp",
    "trip_seconds", "trip_miles", "pickup_community_area",
    "dropoff_community_area", "fare", "tips", "tolls", "extras",
    "trip_total", "payment_type", "company"
)

### Cast trip_seconds and trip_miles to Double Type

In [15]:
from pyspark.sql.types import DoubleType, IntegerType

taxi_df = taxi_df.withColumn("trip_seconds", col("trip_seconds").cast(DoubleType())) \
                 .withColumn("trip_miles", col("trip_miles").cast(DoubleType()))

### Cast community area columns to Integer Type

In [16]:
taxi_df = taxi_df.withColumn("pickup_community_area", col("pickup_community_area").cast(IntegerType())) \
                  .withColumn("dropoff_community_area", col("dropoff_community_area").cast(IntegerType()))

### Display transformed schema and first 5 rows

In [17]:
taxi_df.printSchema()
taxi_df.show(5, truncate=False)

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- company: string (nullable = true)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+--------------------+-------------------+------------+----------+---------------------+----------------------+------+-----+-----+------+----------+------------+-------+


## Stage 5 - Add Derived Columns from Timestamps

### Convert trip_start_timestamp column to proper timestamp

In [18]:
from pyspark.sql.functions import to_timestamp

taxi_df = taxi_df.withColumn(
    "trip_start_timestamp",
    to_timestamp(col("trip_start_timestamp").cast("string"), "yyyy-MM-dd HH:mm:ss")
)

### Extract Year, Month, DayOfWeek, Hour

In [19]:
from pyspark.sql.functions import year, month, dayofweek, hour

taxi_df = taxi_df.withColumn("Year", year(col("trip_start_timestamp"))) \
                  .withColumn("Month", month(col("trip_start_timestamp"))) \
                  .withColumn("DayOfWeek", dayofweek(col("trip_start_timestamp"))) \
                  .withColumn("Hour", hour(col("trip_start_timestamp")))

### Calculate trip duration in minutes

In [20]:
taxi_df = taxi_df.withColumn("trip_duration_minutes", col("trip_seconds") / 60.0)

### Display new schema and 5 rows

In [21]:
taxi_df.printSchema()
taxi_df.show(5, truncate=False)

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- company: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)

+----------------------------------------+-------------------------------------------------------------------------------------------

## Stage 6 - Join with Supplementary Data

### Create supplementary dataframe

In [22]:
from pyspark.sql import Row

community_data = [
    Row(community_area=8, community_name="Near West Side", population=62000),
    Row(community_area=32, community_name="Loop", population=42000),
    Row(community_area=76, community_name="O'Hare", population=13000),
    Row(community_area=6, community_name="Lake View", population=98000),
    Row(community_area=28, community_name="Near North Side", population=85000),
]

community_df = spark.createDataFrame(community_data)
community_df.show()

+--------------+---------------+----------+
|community_area| community_name|population|
+--------------+---------------+----------+
|             8| Near West Side|     62000|
|            32|           Loop|     42000|
|            76|         O'Hare|     13000|
|             6|      Lake View|     98000|
|            28|Near North Side|     85000|
+--------------+---------------+----------+



### Inner join

In [23]:
taxi_with_community_inner = taxi_df.join(
    community_df,
    taxi_df["pickup_community_area"] == community_df["community_area"],
    "inner"
)

inner_row_count = taxi_with_community_inner.count()
print(f"Inner join row count: {inner_row_count}")

Inner join row count: 74938


### Left join and comparison

In [24]:
taxi_with_community_left = taxi_df.join(
    community_df,
    taxi_df["pickup_community_area"] == community_df["community_area"],
    "left"
)

left_row_count = taxi_with_community_left.count()
print(f"Left join row count: {left_row_count}")
print(f"Inner join row count: {inner_row_count}")
print(f"Difference: {left_row_count - inner_row_count} rows")

Left join row count: 124797
Inner join row count: 74938
Difference: 49859 rows


### Assign final output variable and display 5 rows

In [25]:
taxi_with_community = taxi_with_community_left

taxi_with_community.select("trip_id", "pickup_community_area", "community_name").show(5, truncate=False)

+----------------------------------------+---------------------+--------------+
|trip_id                                 |pickup_community_area|community_name|
+----------------------------------------+---------------------+--------------+
|eda829a425ca1da13e1130c700c876881e26ff9a|NULL                 |NULL          |
|0c2b3705c9748a5949bf5b7d727b2d1f2e18805f|NULL                 |NULL          |
|c15dc571c96c36e44bc4ed50f40330e9f0a21650|NULL                 |NULL          |
|3ab618b646a1dc8f6212d762c832a8b0aeac18e7|NULL                 |NULL          |
|3c979de8d901f317228a40673c3923d72104b92c|6                    |Lake View     |
+----------------------------------------+---------------------+--------------+
only showing top 5 rows


### Inner Join vs Left Join Reflection


An **inner join** only keeps rows where the join condition matches on both sides - so a taxi trip is only included if its `pickup_community_area` exists in `community_df`. Since `community_df` only contains 5 specific community areas out of Chicago's 77, an inner join drops every trip whose pickup area isn't one of those 5, resulting in a much smaller row count.

A **left join**, by contrast, keeps every row from the left table (`taxi_df`) regardless of whether a match exists - trips with a matching community area get the extra `community_name`/`population` columns filled in, while trips with no match simply get `null` values for those columns instead of being dropped.

This is why the left join row count equals the original `taxi_df` row count, while the inner join count is much smaller.

## Stage 7 - Aggregations

### Q1 - Top pickup locations

In [26]:
from pyspark.sql.functions import count, avg, desc

top_pickup_locations = taxi_df.groupBy("pickup_community_area") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(desc("trip_count"))

print("Top 10 pickup community areas:")
top_pickup_locations.show(10)

Top 10 pickup community areas:
+---------------------+----------+
|pickup_community_area|trip_count|
+---------------------+----------+
|                    8|     32229|
|                 NULL|     20716|
|                   32|     18549|
|                    6|     11661|
|                    7|      8724|
|                   28|      7695|
|                   24|      5444|
|                   76|      4804|
|                    3|      2639|
|                   22|      2111|
+---------------------+----------+
only showing top 10 rows


### Q2 - Average fare, overall and by payment_type

In [27]:
print("Average trip_total across all trips:")
taxi_df.agg(avg("trip_total").alias("avg_trip_total")).show()

print("Average trip_total by payment_type:")
taxi_df.groupBy("payment_type") \
    .agg(avg("trip_total").alias("avg_trip_total")) \
    .orderBy(desc("avg_trip_total")) \
    .show()

Average trip_total across all trips:
+------------------+
|    avg_trip_total|
+------------------+
|13.551084721588035|
+------------------+

Average trip_total by payment_type:
+------------+------------------+
|payment_type|    avg_trip_total|
+------------+------------------+
|     Unknown|18.977039999999995|
| Credit Card|18.015257683817087|
|     Dispute|14.922075471698115|
|       Pcard| 13.81769230769231|
|   No Charge|12.753155963302751|
|        Cash|12.500331899813084|
+------------+------------------+



### Q3 - Trips by hour

In [28]:
trips_by_hour = taxi_df.groupBy("Hour") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(desc("trip_count"))

print("Trips by hour (sorted by count, highest first):")
trips_by_hour.show(24)


Trips by hour (sorted by count, highest first):
+----+----------+
|Hour|trip_count|
+----+----------+
|   1|      7765|
|   2|      7352|
|  18|      7091|
|  19|      7056|
|   0|      6781|
|  17|      6397|
|  20|      6222|
|   3|      5734|
|  16|      5616|
|  21|      5339|
|   9|      5328|
|  11|      5293|
|  12|      5212|
|  10|      5098|
|  22|      4940|
|  15|      4916|
|  13|      4831|
|   8|      4728|
|  14|      4715|
|  23|      4087|
|   4|      3559|
|   7|      2976|
|   6|      1924|
|   5|      1837|
+----+----------+



### Q4 - Company comparison

In [29]:
company_comparison = taxi_df.groupBy("company") \
    .agg(
        count("*").alias("trip_count"),
        avg("trip_miles").alias("avg_trip_miles")
    ) \
    .orderBy(desc("trip_count"))

print("Top 10 companies by trip count:")
company_comparison.show(10, truncate=False)


Top 10 companies by trip count:
+---------------------------------+----------+------------------+
|company                          |trip_count|avg_trip_miles    |
+---------------------------------+----------+------------------+
|NULL                             |88711     |3.730907441016346 |
|Taxi Affiliation Services        |14437     |18.772064833414483|
|Dispatch Taxi Affiliation        |11547     |3.2542824976184286|
|Blue Ribbon Taxi Association Inc.|4228      |0.2933065279091821|
|Northwest Management LLC         |3656      |3.1562636761488023|
|Choice Taxi Association          |2169      |3.049054863992616 |
|6743 - Luhak Corp                |49        |1.7367346938775512|
+---------------------------------+----------+------------------+



## Stage 8 - Community Area Analysis

### Group by pickup area - trips and average fare, sorted, top 10

In [30]:
community_summary = taxi_df.groupBy("pickup_community_area") \
    .agg(
        count("*").alias("trip_count"),
        avg("fare").alias("avg_fare")
    ) \
    .orderBy(desc("trip_count"))

print("Top 10 community areas by trip count:")
community_summary.show(10)


Top 10 community areas by trip count:
+---------------------+----------+------------------+
|pickup_community_area|trip_count|          avg_fare|
+---------------------+----------+------------------+
|                    8|     32229|10.695579136802152|
|                 NULL|     20716| 11.44686570766556|
|                   32|     18549| 9.162369939080198|
|                    6|     11661|13.327435039876496|
|                    7|      8724|  10.5873223292068|
|                   28|      7695|   9.8482897985705|
|                   24|      5444|13.007755326965484|
|                   76|      4804| 33.59979600333041|
|                    3|      2639|11.289628647214855|
|                   22|      2111| 10.68456655613453|
+---------------------+----------+------------------+
only showing top 10 rows


### Join aggregated result with community_df for names

In [31]:
community_summary_named = community_summary.join(
    community_df,
    community_summary["pickup_community_area"] == community_df["community_area"],
    "left"
)

print("Community area summary with names (top 10 by trip count):")
community_summary_named.select(
    "pickup_community_area", "community_name", "trip_count", "avg_fare"
).orderBy(desc("trip_count")).show(10, truncate=False)

Community area summary with names (top 10 by trip count):
+---------------------+---------------+----------+------------------+
|pickup_community_area|community_name |trip_count|avg_fare          |
+---------------------+---------------+----------+------------------+
|8                    |Near West Side |32229     |10.695579136802152|
|NULL                 |NULL           |20716     |11.44686570766556 |
|32                   |Loop           |18549     |9.162369939080198 |
|6                    |Lake View      |11661     |13.327435039876496|
|7                    |NULL           |8724      |10.5873223292068  |
|28                   |Near North Side|7695      |9.8482897985705   |
|24                   |NULL           |5444      |13.007755326965484|
|76                   |O'Hare         |4804      |33.59979600333041 |
|3                    |NULL           |2639      |11.289628647214855|
|22                   |NULL           |2111      |10.68456655613453 |
+---------------------+---------

### Filter for Community Area 76 - O'Hare

In [32]:
ohare_trip_count = taxi_df.filter(col("pickup_community_area") == 76).count()
print(f"Trips originating in Community Area 76 (O'Hare): {ohare_trip_count}")

Trips originating in Community Area 76 (O'Hare): 4804


### Community Area Analysis Reflection

Area 8 (Near West Side) has the highest trip count by a wide margin at 32,229 trips, followed by Loop (area 32) at 18,549 and Lake View (area 6) at 11,661 - these are all central, high-density areas of Chicago with heavy commercial activity, nightlife, and foot traffic, which naturally drives taxi demand.

Area 76 (O'Hare) stands out differently: despite ranking lower in trip count (4,804), it has by far the highest average fare at $33.60 - more than double every other area in the top 10 - which makes sense given O'Hare is an airport, and airport trips are typically long-distance rides to/from downtown or surrounding suburbs rather than short local hops.

The `NULL` pickup_community_area group itself accounts for 20,716 trips, the second-highest count overall, reflecting the missing geolocation data already noted in Stage 3.

## Stage 9 - Window Functions for Advanced Analysis

### Analysis 1: Rank top 3 companies per community area

In [38]:
from pyspark.sql import Window
from pyspark.sql.functions import rank

company_area_counts = taxi_df.groupBy("pickup_community_area", "company") \
    .agg(count("*").alias("trip_count"))

area_window = Window.partitionBy("pickup_community_area").orderBy(desc("trip_count"))

ranked_companies = company_area_counts.withColumn("rank", rank().over(area_window))

top3_companies_per_area = ranked_companies.filter(col("rank") <= 3)

print("Top 3 companies per pickup community area:")
top3_companies_per_area.orderBy("pickup_community_area", "rank").show(30, truncate=False)


Top 3 companies per pickup community area:
+---------------------+---------------------------------+----------+----+
|pickup_community_area|company                          |trip_count|rank|
+---------------------+---------------------------------+----------+----+
|NULL                 |NULL                             |20077     |1   |
|NULL                 |Dispatch Taxi Affiliation        |256       |2   |
|NULL                 |Taxi Affiliation Services        |172       |3   |
|1                    |NULL                             |318       |1   |
|1                    |Taxi Affiliation Services        |105       |2   |
|1                    |Dispatch Taxi Affiliation        |55        |3   |
|2                    |NULL                             |284       |1   |
|2                    |Taxi Affiliation Services        |48        |2   |
|2                    |Blue Ribbon Taxi Association Inc.|23        |3   |
|3                    |NULL                             |1721      |1

### Analysis 2: lead - next trip start time per company

In [36]:
from pyspark.sql.functions import lead

company_time_window = Window.partitionBy("company").orderBy("trip_start_timestamp")

taxi_with_lead = taxi_df.withColumn(
    "next_trip_start_time",
    lead("trip_start_timestamp").over(company_time_window)
)

print("Next trip start time per company (lead function):")
taxi_with_lead.select("company", "trip_start_timestamp", "next_trip_start_time") \
    .filter(col("company").isNotNull()) \
    .orderBy("company", "trip_start_timestamp") \
    .show(10, truncate=False)

Next trip start time per company (lead function):
+-----------------+--------------------+--------------------+
|company          |trip_start_timestamp|next_trip_start_time|
+-----------------+--------------------+--------------------+
|6743 - Luhak Corp|2013-01-01 00:30:00 |2013-01-01 16:45:00 |
|6743 - Luhak Corp|2013-01-01 16:45:00 |2013-01-01 17:15:00 |
|6743 - Luhak Corp|2013-01-01 17:15:00 |2013-01-01 17:15:00 |
|6743 - Luhak Corp|2013-01-01 17:15:00 |2013-01-01 17:30:00 |
|6743 - Luhak Corp|2013-01-01 17:30:00 |2013-01-01 18:15:00 |
|6743 - Luhak Corp|2013-01-01 18:15:00 |2013-01-01 18:30:00 |
|6743 - Luhak Corp|2013-01-01 18:30:00 |2013-01-02 09:00:00 |
|6743 - Luhak Corp|2013-01-02 09:00:00 |2013-01-02 09:15:00 |
|6743 - Luhak Corp|2013-01-02 09:15:00 |2013-01-02 09:45:00 |
|6743 - Luhak Corp|2013-01-02 09:45:00 |2013-01-02 10:30:00 |
+-----------------+--------------------+--------------------+
only showing top 10 rows


### Analysis 3: lag - year-over-year change per community area

In [43]:
from pyspark.sql.functions import lag

yearly_counts = taxi_df.groupBy("pickup_community_area", "Year") \
    .agg(count("*").alias("trip_count"))

year_window = Window.partitionBy("pickup_community_area").orderBy("Year")

yearly_with_change = yearly_counts.withColumn(
    "previous_year_count", lag("trip_count").over(year_window)
).withColumn(
    "year_over_year_change", col("trip_count") - col("previous_year_count")
)

print("Year-over-year trip count change per community area:")
yearly_with_change.orderBy("pickup_community_area", "Year").show(30, truncate=False)

Year-over-year trip count change per community area:
+---------------------+----+----------+-------------------+---------------------+
|pickup_community_area|Year|trip_count|previous_year_count|year_over_year_change|
+---------------------+----+----------+-------------------+---------------------+
|NULL                 |2013|20716     |NULL               |NULL                 |
|1                    |2013|537       |NULL               |NULL                 |
|2                    |2013|383       |NULL               |NULL                 |
|3                    |2013|2639      |NULL               |NULL                 |
|4                    |2013|804       |NULL               |NULL                 |
|5                    |2013|1128      |NULL               |NULL                 |
|6                    |2013|11661     |NULL               |NULL                 |
|7                    |2013|8724      |NULL               |NULL                 |
|8                    |2013|32229     |NULL  

### Why Window Functions instead of GroupBy


A simple `groupBy` collapses rows into one aggregated row per group, losing row-level detail - it can tell you the total trip count per company, but it can't rank each company *within* its community area while still showing every company's row, or compare one row to the row before/after it.

Window functions solve this using `partitionBy` (defining the group) and `orderBy` (defining the order within that group) without collapsing the DataFrame — each row keeps its identity while gaining context from related rows.

This is visible directly in the results: `rank()` shows the top 3 companies varying meaningfully by community area (e.g. Taxi Affiliation Services and Dispatch Taxi Affiliation dominate most areas, but their exact rank order shifts area to area) — something a plain `groupBy("company")` alone couldn't produce, since it wouldn't be partitioned per area.

Similarly, `lead()` reveals the gap until each company's next trip, useful for understanding fleet utilization patterns, which requires row-to-row comparison, not aggregation. The `lag()` year-over-year analysis returned all `NULL` values in this case, because the 200,000-row sample only contains trips from a single year (2013) - with no prior year present in the data, there's nothing for `lag()` to compare against, which itself illustrates a limitation of working with a small sample rather than the full multi-year dataset.